# OSTrack + V-JEPA re-localization on the dev subset (Kaggle)

Training-free: when OSTrack doubts and V-JEPA sees the target elsewhere, the search region moves there
(`src/tracker/relocalize.py`). Uses the V-JEPA cache made by `kaggle_vjepa.ipynb`.

Before running:
1. Open the finished `kaggle_vjepa` version -> Output -> **New Dataset** (name it e.g. `vjepa-cache-dev`).
2. Here: **Add Input -> Your Datasets -> vjepa-cache-dev**.
3. Settings: **GPU T4 x2**, **Internet = On**. Then **Save Version -> Save & Run All**.

Output: `/kaggle/working/ostrack_vjepa_reloc.zip` (~40 min).

In [ ]:
# fail fast with a clear message if the session has no internet or no cache
import socket, glob, os
try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError:
    raise SystemExit('NO INTERNET: Settings (right panel) -> Internet = On '
                     '(needs a phone-verified Kaggle account), then run again.')
npz = glob.glob('/kaggle/input/**/*.npz', recursive=True)
if not npz:
    raise SystemExit('NO V-JEPA CACHE: Add Input -> the dataset made from the kaggle_vjepa output.')
CACHE = os.path.dirname(npz[0])
print(len(npz), 'cached videos in', CACHE)

In [ ]:
import subprocess  # check=True stops the run here if cloning fails
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/MariiaHamanuk/Object-Tracking.git',
                '/kaggle/working/Object-Tracking'], check=True)
!pip install -q timm==0.6.13 easydict visdom tensorboardX jpeg4py lmdb gdown
!gdown 1dySfPrSk-knAEo0lTXqZ1kMk3BCvVofV -O /kaggle/working/OSTrack_ep0300.pth.tar

In [ ]:
# one process per GPU; each tracks every n-th dev video
import subprocess, torch
os.chdir('/kaggle/working/Object-Tracking')
OUT = '/kaggle/working/results_raw/ostrack_vjepa_reloc/lasot'
n = max(torch.cuda.device_count(), 1)
procs = []
for i in range(n):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
    log = open(f'/kaggle/working/reloc_shard{i}.log', 'w')
    procs.append(subprocess.Popen(
        ['python', 'scripts/run_ostrack_lasot.py', '--checkpoint', '/kaggle/working/OSTrack_ep0300.pth.tar',
         '--sequences', 'configs/splits/dev_occlusion.txt', '--vjepa-cache', CACHE,
         '--out', OUT, '--work', '/kaggle/tmp', '--shard', str(i), '--num-shards', str(n)],
        env=env, stdout=log, stderr=subprocess.STDOUT))
print([p.wait() for p in procs])  # exit codes; 0 = all videos done

In [ ]:
!grep -hE 'FAILED|Error|Traceback|\[[0-9]+/' /kaggle/working/reloc_shard*.log | tail -n 20
!cd /kaggle/working && zip -qr ostrack_vjepa_reloc.zip results_raw reloc_shard*.log && rm -rf Object-Tracking OSTrack_ep0300.pth.tar && ls -la